# YOLOv11 LCN Integrated — Training Notebook
Final training notebook for LCN integrated directly into YOLOv11 architecture.

In [ ]:
#mounting drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#setting up

!pip install ultralytics -q

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from ultralytics import YOLO
from google.colab import drive

**LCN Layer Definition**

In [ ]:
#defining LCN layer as nn.module

class LCNLayer(nn.Module):
    def __init__(self, kernel_size=9, epsilon=1e-6, clip_percentile=1.0):
        super(LCNLayer, self).__init__()
        self.kernel_size = kernel_size
        self.epsilon = epsilon
        self.clip_percentile = clip_percentile

    def forward(self, x):
        # x shape: (B, C, H, W), normalized [0,1] by YOLO
        device = x.device
        results = []

        for i in range(x.shape[0]):  # loop for each image batch
            img = x[i]  # shape: (C, H, W)
            channels = []

            for c in range(img.shape[0]):  # loop per channel (R, G, B)
                ch = img[c].unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)

                # Gaussian blur for local mean deviation
                pad = self.kernel_size // 2
                ch_padded = torch.nn.functional.pad(ch, (pad, pad, pad, pad), mode='reflect')
                kernel = self._gaussian_kernel(self.kernel_size).to(device)
                mu = torch.nn.functional.conv2d(ch_padded, kernel)

                # counting the local std
                diff = ch - mu
                diff_sq_padded = torch.nn.functional.pad(diff ** 2, (pad, pad, pad, pad), mode='reflect')
                sigma = torch.nn.functional.conv2d(diff_sq_padded, kernel)
                sigma = torch.sqrt(sigma + self.epsilon)

                # Normalization
                lcn = diff / (sigma + self.epsilon)

                # Percentile clipping
                lcn_np = lcn.squeeze().detach().cpu().numpy()
                lo = np.percentile(lcn_np, self.clip_percentile)
                hi = np.percentile(lcn_np, 100 - self.clip_percentile)
                lcn_clipped = torch.clamp(lcn.squeeze(), float(lo), float(hi))

                # Rescale to [0, 1]
                lcn_min = lcn_clipped.min()
                lcn_max = lcn_clipped.max()
                lcn_norm = (lcn_clipped - lcn_min) / (lcn_max - lcn_min + self.epsilon)
                channels.append(lcn_norm)

            results.append(torch.stack(channels, dim=0))

        return torch.stack(results, dim=0)

    def _gaussian_kernel(self, size):
        # for 2D Gaussian kernel
        sigma = size / 6.0
        coords = torch.arange(size).float() - size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = g / g.sum()
        kernel_2d = g.outer(g)
        kernel_2d = kernel_2d / kernel_2d.sum()
        return kernel_2d.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)


In [ ]:
#making path to dataset

DATASET_PATH = '/content/drive/MyDrive/YOUR_DATASET_FOLDER/'

# checking dataset path
for split in ['train', 'val', 'test']:
    img_path = os.path.join(DATASET_PATH, 'images', split)
    lbl_path = os.path.join(DATASET_PATH, 'labels', split)
    img_count = len(os.listdir(img_path)) if os.path.exists(img_path) else 0
    lbl_count = len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0
    print(f"{split:5s} → images: {img_count:4d} | labels: {lbl_count:4d}")

# Path model & output
MODEL_WEIGHTS = 'yolov11m.pt' #models
PROJECT_DIR   = '/content/drive/MyDrive/YOLO/runs'
EXP_NAME      = 'yolov11_lcn_integrated'

**Model Setup & Training**

In [ ]:
#making yaml file for training

import yaml

data_config = {
    'path': DATASET_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 7,
    'names': ['car', 'motorcycle', 'bus', 'motorized-thrisaws', 'truck', 'person', 'traffic sign']
} #depends on the dataset

yaml_path = '/content/drive/MyDrive/YOLO/data_integrated.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

In [ ]:
# Load model
model_path = os.path.join(os.path.dirname(PROJECT_DIR), MODEL_WEIGHTS)

# To check if model file exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model weights not found at: {model_path}. Please ensure the file exists in your Google Drive at this exact location.")

model = YOLO(model_path)

# LCNLayer
lcn_layer = LCNLayer(kernel_size=9, epsilon=1e-6, clip_percentile=1.0)

# Patch forward pass model internal
original_forward = model.model.forward

def lcn_forward(x, *args, **kwargs):
    x = lcn_layer.to(x.device)(x)
    return original_forward(x, *args, **kwargs)

model.model.forward = lcn_forward

# Unfreeze all parameter
for param in model.model.parameters():
    param.requires_grad = True

In [ ]:
#train

import json
from datetime import datetime

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=20,
    freeze=None,
    project=PROJECT_DIR,
    name=EXP_NAME,
    exist_ok=True,
    resume=True,
    verbose=True
)

In [ ]:
# test

best_model_path = f"{PROJECT_DIR}/{EXP_NAME}/weights/best.pt"
test_model = YOLO(best_model_path)

# Patch LCN to test model
original_forward_test = test_model.model.forward
def lcn_forward_test(x, *args, **kwargs):
    x = lcn_layer.to(x.device)(x)
    return original_forward_test(x, *args, **kwargs)
test_model.model.forward = lcn_forward_test

# Run
test_results = test_model.val(
    data=yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    project=PROJECT_DIR,
    name=f"{EXP_NAME}_test",
    exist_ok=True,
    verbose=True
)

# metrik
map50     = test_results.results_dict['metrics/mAP50(B)']
map5095   = test_results.results_dict['metrics/mAP50-95(B)']
precision = test_results.results_dict['metrics/precision(B)']
recall    = test_results.results_dict['metrics/recall(B)']


In [ ]:
#save

summary = {
    'experiment': EXP_NAME,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model': MODEL_WEIGHTS,
    'dataset': DATASET_PATH,
    'config': {
        'batch': 16,
        'optimizer': 'AdamW',
        'lr0': 0.001,
        'freeze': None,
        'lcn_kernel': 9,
        'clip_percentile': 1.0
    },
    'val_results': {
        'mAP50': round(results.results_dict['metrics/mAP50(B)'], 4),
        'mAP50_95': round(results.results_dict['metrics/mAP50-95(B)'], 4),
    },
    'test_results': {
        'mAP50': round(map50, 4),
        'mAP50_95': round(map5095, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
    }
}

save_path = f"{PROJECT_DIR}/{EXP_NAME}/results_summary.json"
with open(save_path, 'w') as f:
    json.dump(summary, f, indent=4)